# 1) Path to Raw and Processed Folder:

In [0]:
%run ../common/configuration


In [0]:
# dbutils.widgets.text("raw_folder_path", "")
# dbutils.widgets.text("processed_folder_path", "")
 
# raw_folder_path = dbutils.widgets.get("raw_folder_path")
# processed_folder_path = dbutils.widgets.get("processed_folder_path")

# 2) Process circuits.json file:

In [0]:
import json

def load_json_from_raw(relative_path):
    full_path = f"{raw_folder_path}/{relative_path}"
    raw_text = dbutils.fs.head(full_path, 1024 * 1024 * 50)  # up to 50MB
    return json.loads(raw_text)


circuits_payload = load_json_from_raw("circuits/circuits.json")
circuits_records = circuits_payload["MRData"]["CircuitTable"]["Circuits"]

flattened_circuits = [
    {
        "circuit_id": c.get("circuitId"),
        "circuit_name": c.get("circuitName"),
        "url": c.get("url"),
        "lat": c.get("Location", {}).get("lat"),
        "long": c.get("Location", {}).get("long"),
        "locality": c.get("Location", {}).get("locality"),
        "country": c.get("Location", {}).get("country"),
    }
    for c in circuits_records
]

print(f"Flattened {len(flattened_circuits)} circuits.")

In [0]:
import csv
import io

fieldnames = ["circuit_id", "circuit_name", "url", "lat", "long", "locality", "country"]

csv_buffer = io.StringIO()
writer = csv.DictWriter(csv_buffer, fieldnames=fieldnames)
writer.writeheader()
writer.writerows(flattened_circuits)

output_path = f"{processed_folder_path}/circuits/csv/circuits.csv"
dbutils.fs.put(output_path, csv_buffer.getvalue(), overwrite=True)
print(f"saved {output_path}")

In [0]:
circuits_csv_df = spark.read.option("header", True).csv(output_path)
display(circuits_csv_df)

# 3) Process constructor_standings.json:

In [0]:
constructor_standings_payload = load_json_from_raw("constructor_standings/constructor_standings.json")
standings_lists = constructor_standings_payload["MRData"]["StandingsTable"]["StandingsLists"]

flattened_constructor_standings = []
for standings_list in standings_lists:
    season = standings_list.get("season")
    round_no = standings_list.get("round")
    for entry in standings_list.get("ConstructorStandings", []):
        constructor = entry.get("Constructor", {})
        flattened_constructor_standings.append({
            "season": season,
            "round": round_no,
            "position": entry.get("position"),
            "position_text": entry.get("positionText"),
            "points": entry.get("points"),
            "wins": entry.get("wins"),
            "constructor_id": constructor.get("constructorId"),
            "constructor_name": constructor.get("name"),
            "nationality": constructor.get("nationality"),
            "url": constructor.get("url"),
        })

print(f"Flattened {len(flattened_constructor_standings)} constructor-standing rows across {len(standings_lists)} season(s).")

In [0]:
cs_fieldnames = ["season", "round", "position", "position_text", "points", "wins",
                  "constructor_id", "constructor_name", "nationality", "url"]

cs_csv_buffer = io.StringIO()
cs_writer = csv.DictWriter(cs_csv_buffer, fieldnames=cs_fieldnames)
cs_writer.writeheader()
cs_writer.writerows(flattened_constructor_standings)

cs_output_path = f"{processed_folder_path}/constructor_standings/csv/constructor_standings.csv"
dbutils.fs.put(cs_output_path, cs_csv_buffer.getvalue(), overwrite=True)
print(f"saved {cs_output_path}")

In [0]:
constructor_standings_csv_df = spark.read.option("header", True).csv(cs_output_path)
display(constructor_standings_csv_df)

# 4) Process constructors.json:

In [0]:
constructors_payload = load_json_from_raw("constructors/constructors.json")
constructor_records = constructors_payload["MRData"]["ConstructorTable"]["Constructors"]

seen_constructor_ids = set()
flattened_constructors = []
for c in constructor_records:
    constructor_id = c.get("constructorId")
    if constructor_id in seen_constructor_ids:
        continue
    seen_constructor_ids.add(constructor_id)
    flattened_constructors.append({
        "constructor_id": constructor_id,
        "name": c.get("name"),
        "nationality": c.get("nationality"),
        "url": c.get("url"),
    })

print(f"Flattened {len(flattened_constructors)} unique constructors (from {len(constructor_records)} raw rows).")

In [0]:
constructors_fieldnames = ["constructor_id", "name", "nationality", "url"]

constructors_csv_buffer = io.StringIO()
constructors_writer = csv.DictWriter(constructors_csv_buffer, fieldnames=constructors_fieldnames)
constructors_writer.writeheader()
constructors_writer.writerows(flattened_constructors)

constructors_output_path = f"{processed_folder_path}/constructors/csv/constructors.csv"
dbutils.fs.put(constructors_output_path, constructors_csv_buffer.getvalue(), overwrite=True)
print(f"saved {constructors_output_path}")

In [0]:
constructors_csv_df = spark.read.option("header", True).csv(constructors_output_path)
display(constructors_csv_df)

# 5) Process driver_standings.json:

In [0]:
driver_standings_payload = load_json_from_raw("driver_standings/driver_standings.json")
driver_standings_lists = driver_standings_payload["MRData"]["StandingsTable"]["StandingsLists"]

flattened_driver_standings = []
for standings_list in driver_standings_lists:
    season = standings_list.get("season")
    round_no = standings_list.get("round")
    for entry in standings_list.get("DriverStandings", []):
        driver = entry.get("Driver", {})
        constructor_ids = "|".join(c.get("constructorId", "") for c in entry.get("Constructors", []))
        constructor_names = "|".join(c.get("name", "") for c in entry.get("Constructors", []))
        flattened_driver_standings.append({
            "season": season,
            "round": round_no,
            "position": entry.get("position"),
            "position_text": entry.get("positionText"),
            "points": entry.get("points"),
            "wins": entry.get("wins"),
            "driver_id": driver.get("driverId"),
            "permanent_number": driver.get("permanentNumber"),
            "code": driver.get("code"),
            "given_name": driver.get("givenName"),
            "family_name": driver.get("familyName"),
            "date_of_birth": driver.get("dateOfBirth"),
            "nationality": driver.get("nationality"),
            "constructor_ids": constructor_ids,
            "constructor_names": constructor_names,
        })

print(f"Flattened {len(flattened_driver_standings)} driver-standing rows across {len(driver_standings_lists)} season(s).")

In [0]:
ds_fieldnames = ["season", "round", "position", "position_text", "points", "wins",
                  "driver_id", "permanent_number", "code", "given_name", "family_name",
                  "date_of_birth", "nationality", "constructor_ids", "constructor_names"]

ds_csv_buffer = io.StringIO()
ds_writer = csv.DictWriter(ds_csv_buffer, fieldnames=ds_fieldnames)
ds_writer.writeheader()
ds_writer.writerows(flattened_driver_standings)

ds_output_path = f"{processed_folder_path}/driver_standings/csv/driver_standings.csv"
dbutils.fs.put(ds_output_path, ds_csv_buffer.getvalue(), overwrite=True)
print(f"saved {ds_output_path}")

In [0]:
driver_standings_csv_df = spark.read.option("header", True).csv(ds_output_path)
display(driver_standings_csv_df)

# 6) Process drivers.json:

In [0]:
drivers_payload = load_json_from_raw("drivers/drivers.json")
driver_records = drivers_payload["MRData"]["DriverTable"]["Drivers"]

seen_driver_ids = set()
flattened_drivers = []
for d in driver_records:
    driver_id = d.get("driverId")
    if driver_id in seen_driver_ids:
        continue
    seen_driver_ids.add(driver_id)
    flattened_drivers.append({
        "driver_id": driver_id,
        "permanent_number": d.get("permanentNumber"),
        "code": d.get("code"),
        "given_name": d.get("givenName"),
        "family_name": d.get("familyName"),
        "date_of_birth": d.get("dateOfBirth"),
        "nationality": d.get("nationality"),
        "url": d.get("url"),
    })

print(f"Flattened {len(flattened_drivers)} unique drivers (from {len(driver_records)} raw rows).")

In [0]:
drivers_fieldnames = ["driver_id", "permanent_number", "code", "given_name", "family_name",
                       "date_of_birth", "nationality", "url"]

drivers_csv_buffer = io.StringIO()
drivers_writer = csv.DictWriter(drivers_csv_buffer, fieldnames=drivers_fieldnames)
drivers_writer.writeheader()
drivers_writer.writerows(flattened_drivers)

drivers_output_path = f"{processed_folder_path}/drivers/csv/drivers.csv"
dbutils.fs.put(drivers_output_path, drivers_csv_buffer.getvalue(), overwrite=True)
print(f"saved {drivers_output_path}")

In [0]:
drivers_csv_df = spark.read.option("header", True).csv(drivers_output_path)
display(drivers_csv_df)

# 7) Process lap_times.json:

In [0]:
lap_times_payload = load_json_from_raw("lap_times/lap_times.json")
race_entries = lap_times_payload["MRData"]["RaceTable"]["Races"]

flattened_lap_times = []
for race in race_entries:
    season = race.get("season")
    round_no = race.get("round")
    race_name = race.get("raceName")
    circuit_id = race.get("Circuit", {}).get("circuitId")
    for lap in race.get("Laps", []):
        lap_number = lap.get("number")
        for timing in lap.get("Timings", []):
            flattened_lap_times.append({
                "season": season,
                "round": round_no,
                "race_name": race_name,
                "circuit_id": circuit_id,
                "lap": lap_number,
                "driver_id": timing.get("driverId"),
                "position": timing.get("position"),
                "time": timing.get("time"),
            })

print(f"Flattened {len(flattened_lap_times)} lap-time rows across {len(race_entries)} race-page entries.")

In [0]:
lap_times_fieldnames = ["season", "round", "race_name", "circuit_id", "lap", "driver_id", "position", "time"]

lap_times_csv_buffer = io.StringIO()
lap_times_writer = csv.DictWriter(lap_times_csv_buffer, fieldnames=lap_times_fieldnames)
lap_times_writer.writeheader()
lap_times_writer.writerows(flattened_lap_times)

lap_times_output_path = f"{processed_folder_path}/lap_times/csv/lap_times.csv"
dbutils.fs.put(lap_times_output_path, lap_times_csv_buffer.getvalue(), overwrite=True)
print(f"saved {lap_times_output_path}")

In [0]:
lap_times_csv_df = spark.read.option("header", True).csv(lap_times_output_path)
display(lap_times_csv_df)

# 8) Process pit_stops.json:

In [0]:
pit_stops_payload = load_json_from_raw("pit_stops/pit_stops.json")
pit_stop_race_entries = pit_stops_payload["MRData"]["RaceTable"]["Races"]

flattened_pit_stops = []
for race in pit_stop_race_entries:
    season = race.get("season")
    round_no = race.get("round")
    race_name = race.get("raceName")
    circuit_id = race.get("Circuit", {}).get("circuitId")
    for stop in race.get("PitStops", []):
        flattened_pit_stops.append({
            "season": season,
            "round": round_no,
            "race_name": race_name,
            "circuit_id": circuit_id,
            "driver_id": stop.get("driverId"),
            "lap": stop.get("lap"),
            "stop": stop.get("stop"),
            "time": stop.get("time"),
            "duration": stop.get("duration"),
        })

print(f"Flattened {len(flattened_pit_stops)} pit-stop rows across {len(pit_stop_race_entries)} race-page entries.")

In [0]:
pit_stops_fieldnames = ["season", "round", "race_name", "circuit_id", "driver_id", "lap", "stop", "time", "duration"]

pit_stops_csv_buffer = io.StringIO()
pit_stops_writer = csv.DictWriter(pit_stops_csv_buffer, fieldnames=pit_stops_fieldnames)
pit_stops_writer.writeheader()
pit_stops_writer.writerows(flattened_pit_stops)

pit_stops_output_path = f"{processed_folder_path}/pit_stops/csv/pit_stops.csv"
dbutils.fs.put(pit_stops_output_path, pit_stops_csv_buffer.getvalue(), overwrite=True)
print(f"saved {pit_stops_output_path}")

In [0]:
pit_stops_csv_df = spark.read.option("header", True).csv(pit_stops_output_path)
display(pit_stops_csv_df)

# 9) Process qualifying.json:

In [0]:
qualifying_payload = load_json_from_raw("qualifying/qualifying.json")
qualifying_race_entries = qualifying_payload["MRData"]["RaceTable"]["Races"]

flattened_qualifying = []
for race in qualifying_race_entries:
    season = race.get("season")
    round_no = race.get("round")
    race_name = race.get("raceName")
    circuit_id = race.get("Circuit", {}).get("circuitId")
    for result in race.get("QualifyingResults", []):
        driver = result.get("Driver", {})
        constructor = result.get("Constructor", {})
        flattened_qualifying.append({
            "season": season,
            "round": round_no,
            "race_name": race_name,
            "circuit_id": circuit_id,
            "number": result.get("number"),
            "position": result.get("position"),
            "driver_id": driver.get("driverId"),
            "code": driver.get("code"),
            "given_name": driver.get("givenName"),
            "family_name": driver.get("familyName"),
            "nationality": driver.get("nationality"),
            "constructor_id": constructor.get("constructorId"),
            "constructor_name": constructor.get("name"),
            "q1": result.get("Q1"),
            "q2": result.get("Q2"),
            "q3": result.get("Q3"),
        })

print(f"Flattened {len(flattened_qualifying)} qualifying rows across {len(qualifying_race_entries)} race-page entries.")

In [0]:
qualifying_fieldnames = ["season", "round", "race_name", "circuit_id", "number", "position",
                          "driver_id", "code", "given_name", "family_name", "nationality",
                          "constructor_id", "constructor_name", "q1", "q2", "q3"]

qualifying_csv_buffer = io.StringIO()
qualifying_writer = csv.DictWriter(qualifying_csv_buffer, fieldnames=qualifying_fieldnames)
qualifying_writer.writeheader()
qualifying_writer.writerows(flattened_qualifying)

qualifying_output_path = f"{processed_folder_path}/qualifying/csv/qualifying.csv"
dbutils.fs.put(qualifying_output_path, qualifying_csv_buffer.getvalue(), overwrite=True)
print(f"saved {qualifying_output_path}")

In [0]:
qualifying_csv_df = spark.read.option("header", True).csv(qualifying_output_path)
display(qualifying_csv_df)

# 9) Process races.json:

In [0]:
races_payload = load_json_from_raw("races/races.json")
race_entries = races_payload["MRData"]["RaceTable"]["Races"]

def session_field(race, session_key, field):
    return race.get(session_key, {}).get(field)

flattened_races = []
for race in race_entries:
    circuit = race.get("Circuit", {})
    location = circuit.get("Location", {})
    flattened_races.append({
        "season": race.get("season"),
        "round": race.get("round"),
        "race_name": race.get("raceName"),
        "url": race.get("url"),
        "date": race.get("date"),
        "time": race.get("time"),
        "circuit_id": circuit.get("circuitId"),
        "circuit_name": circuit.get("circuitName"),
        "lat": location.get("lat"),
        "long": location.get("long"),
        "locality": location.get("locality"),
        "country": location.get("country"),
        "first_practice_date": session_field(race, "FirstPractice", "date"),
        "first_practice_time": session_field(race, "FirstPractice", "time"),
        "second_practice_date": session_field(race, "SecondPractice", "date"),
        "second_practice_time": session_field(race, "SecondPractice", "time"),
        "third_practice_date": session_field(race, "ThirdPractice", "date"),
        "third_practice_time": session_field(race, "ThirdPractice", "time"),
        "qualifying_date": session_field(race, "Qualifying", "date"),
        "qualifying_time": session_field(race, "Qualifying", "time"),
        "sprint_date": session_field(race, "Sprint", "date"),
        "sprint_time": session_field(race, "Sprint", "time"),
    })

print(f"Flattened {len(flattened_races)} races.")

In [0]:
races_fieldnames = ["season", "round", "race_name", "url", "date", "time",
                     "circuit_id", "circuit_name", "lat", "long", "locality", "country",
                     "first_practice_date", "first_practice_time",
                     "second_practice_date", "second_practice_time",
                     "third_practice_date", "third_practice_time",
                     "qualifying_date", "qualifying_time",
                     "sprint_date", "sprint_time"]

races_csv_buffer = io.StringIO()
races_writer = csv.DictWriter(races_csv_buffer, fieldnames=races_fieldnames)
races_writer.writeheader()
races_writer.writerows(flattened_races)

races_output_path = f"{processed_folder_path}/races/csv/races.csv"
dbutils.fs.put(races_output_path, races_csv_buffer.getvalue(), overwrite=True)
print(f"saved {races_output_path}")

In [0]:
races_csv_df = spark.read.option("header", True).csv(races_output_path)
display(races_csv_df)

# 11) Process results.json:

In [0]:
results_payload = load_json_from_raw("results/results.json")
results_race_entries = results_payload["MRData"]["RaceTable"]["Races"]

flattened_results = []
for race in results_race_entries:
    season = race.get("season")
    round_no = race.get("round")
    race_name = race.get("raceName")
    circuit_id = race.get("Circuit", {}).get("circuitId")
    for result in race.get("Results", []):
        driver = result.get("Driver", {})
        constructor = result.get("Constructor", {})
        time_info = result.get("Time", {})
        fastest_lap = result.get("FastestLap", {})
        fastest_lap_time = fastest_lap.get("Time", {})
        fastest_lap_speed = fastest_lap.get("AverageSpeed", {})
        flattened_results.append({
            "season": season,
            "round": round_no,
            "race_name": race_name,
            "circuit_id": circuit_id,
            "number": result.get("number"),
            "position": result.get("position"),
            "position_text": result.get("positionText"),
            "points": result.get("points"),
            "grid": result.get("grid"),
            "laps": result.get("laps"),
            "status": result.get("status"),
            "driver_id": driver.get("driverId"),
            "code": driver.get("code"),
            "given_name": driver.get("givenName"),
            "family_name": driver.get("familyName"),
            "nationality": driver.get("nationality"),
            "constructor_id": constructor.get("constructorId"),
            "constructor_name": constructor.get("name"),
            "time_millis": time_info.get("millis"),
            "time_gap": time_info.get("time"),
            "fastest_lap_rank": fastest_lap.get("rank"),
            "fastest_lap_number": fastest_lap.get("lap"),
            "fastest_lap_time": fastest_lap_time.get("time"),
            "fastest_lap_speed_units": fastest_lap_speed.get("units"),
            "fastest_lap_speed": fastest_lap_speed.get("speed"),
        })

print(f"Flattened {len(flattened_results)} result rows across {len(results_race_entries)} race-page entries.")

In [0]:
results_fieldnames = ["season", "round", "race_name", "circuit_id", "number", "position", "position_text",
                       "points", "grid", "laps", "status", "driver_id", "code", "given_name", "family_name",
                       "nationality", "constructor_id", "constructor_name", "time_millis", "time_gap",
                       "fastest_lap_rank", "fastest_lap_number", "fastest_lap_time",
                       "fastest_lap_speed_units", "fastest_lap_speed"]

results_csv_buffer = io.StringIO()
results_writer = csv.DictWriter(results_csv_buffer, fieldnames=results_fieldnames)
results_writer.writeheader()
results_writer.writerows(flattened_results)

results_output_path = f"{processed_folder_path}/results/csv/results.csv"
dbutils.fs.put(results_output_path, results_csv_buffer.getvalue(), overwrite=True)
print(f"saved {results_output_path}")

In [0]:
results_csv_df = spark.read.option("header", True).csv(results_output_path)
display(results_csv_df)

# 12) Process seasons.json:

In [0]:
seasons_payload = load_json_from_raw("seasons/seasons.json")
season_records = seasons_payload["MRData"]["SeasonTable"]["Seasons"]

flattened_seasons = [
    {
        "season": s.get("season"),
        "url": s.get("url"),
    }
    for s in season_records
]

print(f"Flattened {len(flattened_seasons)} seasons.")

In [0]:
seasons_fieldnames = ["season", "url"]

seasons_csv_buffer = io.StringIO()
seasons_writer = csv.DictWriter(seasons_csv_buffer, fieldnames=seasons_fieldnames)
seasons_writer.writeheader()
seasons_writer.writerows(flattened_seasons)

seasons_output_path = f"{processed_folder_path}/seasons/csv/seasons.csv"
dbutils.fs.put(seasons_output_path, seasons_csv_buffer.getvalue(), overwrite=True)
print(f"saved {seasons_output_path}")

In [0]:
seasons_csv_df = spark.read.option("header", True).csv(seasons_output_path)
display(seasons_csv_df)

# 13) Process sprint.json:

In [0]:
sprint_payload = load_json_from_raw("sprint/sprint.json")
sprint_race_entries = sprint_payload["MRData"]["RaceTable"]["Races"]

flattened_sprint = []
for race in sprint_race_entries:
    season = race.get("season")
    round_no = race.get("round")
    race_name = race.get("raceName")
    circuit_id = race.get("Circuit", {}).get("circuitId")
    for result in race.get("SprintResults", []):
        driver = result.get("Driver", {})
        constructor = result.get("Constructor", {})
        time_info = result.get("Time", {})
        fastest_lap = result.get("FastestLap", {})
        fastest_lap_time = fastest_lap.get("Time", {})
        flattened_sprint.append({
            "season": season,
            "round": round_no,
            "race_name": race_name,
            "circuit_id": circuit_id,
            "number": result.get("number"),
            "position": result.get("position"),
            "position_text": result.get("positionText"),
            "points": result.get("points"),
            "grid": result.get("grid"),
            "laps": result.get("laps"),
            "status": result.get("status"),
            "driver_id": driver.get("driverId"),
            "code": driver.get("code"),
            "given_name": driver.get("givenName"),
            "family_name": driver.get("familyName"),
            "nationality": driver.get("nationality"),
            "constructor_id": constructor.get("constructorId"),
            "constructor_name": constructor.get("name"),
            "time_millis": time_info.get("millis"),
            "time_gap": time_info.get("time"),
            "fastest_lap_number": fastest_lap.get("lap"),
            "fastest_lap_time": fastest_lap_time.get("time"),
        })

print(f"Flattened {len(flattened_sprint)} sprint-result rows across {len(sprint_race_entries)} race-page entries.")

In [0]:
sprint_fieldnames = ["season", "round", "race_name", "circuit_id", "number", "position", "position_text",
                      "points", "grid", "laps", "status", "driver_id", "code", "given_name", "family_name",
                      "nationality", "constructor_id", "constructor_name", "time_millis", "time_gap",
                      "fastest_lap_number", "fastest_lap_time"]

sprint_csv_buffer = io.StringIO()
sprint_writer = csv.DictWriter(sprint_csv_buffer, fieldnames=sprint_fieldnames)
sprint_writer.writeheader()
sprint_writer.writerows(flattened_sprint)

sprint_output_path = f"{processed_folder_path}/sprint/csv/sprint.csv"
dbutils.fs.put(sprint_output_path, sprint_csv_buffer.getvalue(), overwrite=True)
print(f"saved {sprint_output_path}")

In [0]:
sprint_csv_df = spark.read.option("header", True).csv(sprint_output_path)
display(sprint_csv_df)

In [0]:
status_payload = load_json_from_raw("status/status.json")
status_records = status_payload["MRData"]["StatusTable"]["Status"]

flattened_status = [
    {
        "status_id": s.get("statusId"),
        "status": s.get("status"),
        "count": s.get("count"),
    }
    for s in status_records
]

print(f"Flattened {len(flattened_status)} statuses.")

In [0]:
status_fieldnames = ["status_id", "status", "count"]

status_csv_buffer = io.StringIO()
status_writer = csv.DictWriter(status_csv_buffer, fieldnames=status_fieldnames)
status_writer.writeheader()
status_writer.writerows(flattened_status)

status_output_path = f"{processed_folder_path}/status/csv/status.csv"
dbutils.fs.put(status_output_path, status_csv_buffer.getvalue(), overwrite=True)
print(f"saved {status_output_path}")



In [0]:
status_csv_df = spark.read.option("header", True).csv(status_output_path)
display(status_csv_df)